In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from painnet import config, windows, splits, models, evaluate, watch
import lightgbm as lgb

print(config.describe())

painnet | env=local | data=found | C:\Users\raj28\multimodal-pain-assesment\data\raw\PhysioPain Dataset\Multimodal Pain Dataset


In [12]:
dc = watch.duration_check()
dc.head(20)

Loaded watch data with shape: (440428, 9)
Participants: 86
Subjects in both tiers: 77
Median difference in duration (seconds): 27.0


,eeg_seconds,watch_seconds,diff,percent
subject_id,,,,
S062,1307,664.00,643.00,96.84
S042,856,1219.50,-363.50,42.46
S079,911,1290.00,-379.00,41.60
S072,740,1022.00,-282.00,38.11
S066,1204,1583.75,-379.75,31.54
S041,1019,1336.00,-317.00,31.11
S078,1826,1409.75,416.25,29.53
S054,1201,1518.00,-317.00,26.39
S071,1213,1461.00,-248.00,20.45


In [ ]:
X, y, groups = watch.build_watch_dataset()
print(X.shape, len(set(groups)), "subjects")

Loaded watch data with shape: (440428, 9)
Participants: 86
Built watch dataset with 7043 windows, shape: (7043, 240, 6), labels: ['back_pain' 'headache' 'menstrual_pain' 'no_pain']
participants: ['S004' 'S005' 'S006' 'S008' 'S009' 'S010' 'S011' 'S012' 'S015' 'S016'
 'S017' 'S018' 'S019' 'S020' 'S021' 'S022' 'S023' 'S024' 'S025' 'S026'
 'S028' 'S029' 'S030' 'S031' 'S033' 'S034' 'S035' 'S036' 'S037' 'S038'
 'S039' 'S040' 'S041' 'S042' 'S043' 'S044' 'S045' 'S046' 'S047' 'S048'
 'S049' 'S052' 'S053' 'S054' 'S055' 'S056' 'S057' 'S058' 'S059' 'S060'
 'S061' 'S062' 'S063' 'S064' 'S065' 'S066' 'S067' 'S068' 'S069' 'S070'
 'S071' 'S072' 'S073' 'S074' 'S075' 'S076' 'S077' 'S078' 'S079' 'S081'
 'S082' 'S083' 'S084' 'S085' 'S086' 'S087' 'S088' 'S089' 'S090' 'S091'
 'S092' 'S093' 'S094' 'S095' 'S096' 'S097']
(7043, 240, 6) 86 subjects


In [14]:
y_int = windows.encode_labels(y)
Xt = windows.to_tabular(X)

rows = []
for tr, te in splits.subject_folds(groups, y_int):
    m = lgb.LGBMClassifier(n_estimators=300, verbose=-1)
    m.fit(Xt[tr], y_int[tr])
    rows.append(evaluate.fold_metrics(y_int[te], m.predict(Xt[te])))

print(evaluate.aggregate(rows)["report"])

macro_f1        0.231 ± 0.015
weighted_f1     0.280 ± 0.049
balanced_acc    0.264 ± 0.031
accuracy        0.283 ± 0.035
Name: report, dtype: object


In [15]:
Xe, Xw, y_f, groups_f = watch.build_fusion_dataset()
print(Xe.shape, Xw.shape, len(set(groups_f)), "subjects")

Loaded watch data with shape: (440428, 9)
Participants: 86
EEG subjects: 83 Watch subjects: 86 Common subjects: 77
Pain windows:  5936
X_eeg shape: (5936, 60, 10)
X_watch shape: (5936, 240, 6)
Class Participants Windows
no_pain                    15     1143
headache                   27     2103
back_pain                  25     1882
menstrual_pain             10      808
(5936, 60, 10) (5936, 240, 6) 77 subjects
